# PART 1 — KiTS23 Dataset Loading + Preprocessing + Class Analysis

## Research Project:
**3D Kidney and Kidney Tumor Segmentation Using Deep Learning: A Comparative Study on the KiTS23 Dataset**

This notebook section performs:

- Dataset path configuration
- KiTS23 case loading
- 489 training / 110 testing split
- CT image loading
- Segmentation mask loading
- CT normalization
- Label verification
- Class distribution analysis
- KiTS23 class distribution graph
- CT and Ground Truth visualization



In [ ]:

# Import Libraries

import os
import glob
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import nibabel as nib

from tqdm import tqdm

from scipy.ndimage import zoom

print("Libraries Loaded Successfully")


In [ ]:

# Project Paths

PROJECT_PATH = r"C:\Users\USER\Kidney_Segmentation_Project"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data"
)

RESULT_PATH = os.path.join(
    PROJECT_PATH,
    "results"
)

FIGURE_PATH = os.path.join(
    PROJECT_PATH,
    "figures"
)

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "processed_data"
)

os.makedirs(RESULT_PATH, exist_ok=True)
os.makedirs(FIGURE_PATH, exist_ok=True)
os.makedirs(PROCESSED_PATH, exist_ok=True)

print(PROJECT_PATH)


In [ ]:

# Find KiTS23 Cases

cases = sorted(
    glob.glob(
        os.path.join(
            DATA_PATH,
            "case_*"
        )
    )
)

print("Total Cases Found:", len(cases))


In [ ]:

# Train/Test Split

train_cases = cases[:489]

test_cases = cases[489:]

print("Training Cases:", len(train_cases))
print("Testing Cases:", len(test_cases))


In [ ]:

# NIFTI Loading Function

def load_nifti(file_path):

    image = nib.load(file_path)

    volume = image.get_fdata()

    return volume


In [ ]:

# Load Sample Case

sample_case = train_cases[0]

image_file = os.path.join(
    sample_case,
    "imaging.nii.gz"
)

mask_file = os.path.join(
    sample_case,
    "segmentation.nii.gz"
)

ct_image = load_nifti(image_file)

mask = load_nifti(mask_file)

print("CT Shape:", ct_image.shape)
print("Mask Shape:", mask.shape)


In [ ]:

# CT Normalization

def normalize_ct(volume):

    volume = np.clip(
        volume,
        -200,
        300
    )

    volume = (
        volume - np.min(volume)
    ) / (
        np.max(volume)
        -
        np.min(volume)
        +
        1e-8
    )

    return volume


ct_image = normalize_ct(ct_image)

print(
    ct_image.min(),
    ct_image.max()
)


In [ ]:

# Check Segmentation Labels

labels = np.unique(mask)

print("Segmentation Labels:")
print(labels)


In [ ]:

# Class Distribution Analysis

classes = {
    0:"Background",
    1:"Kidney",
    2:"Tumour",
    3:"Cyst"
}

distribution = {
    0:0,
    1:0,
    2:0,
    3:0
}


for case in tqdm(train_cases):

    mask_path = os.path.join(
        case,
        "segmentation.nii.gz"
    )

    mask_volume = load_nifti(mask_path)

    unique, counts = np.unique(
        mask_volume,
        return_counts=True
    )

    for u,c in zip(unique,counts):

        distribution[int(u)] += c


distribution


In [ ]:

# Create Distribution Table

df_distribution = pd.DataFrame(
    {
        "Class":[classes[i] for i in distribution.keys()],
        "Voxel Count":[distribution[i] for i in distribution.keys()]
    }
)

df_distribution


In [ ]:

# KiTS23 Class Distribution Graph

plt.figure(figsize=(8,5))

plt.bar(
    df_distribution["Class"],
    df_distribution["Voxel Count"]
)

plt.title(
    "KiTS23 Class Distribution Analysis",
    fontsize=16
)

plt.xlabel("Segmentation Class")

plt.ylabel("Number of Voxels")

plt.xticks(rotation=30)

plt.grid(axis="y")

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "KiTS23_Class_Distribution.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# CT + Ground Truth Visualization

slice_id = ct_image.shape[2]//2

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)

plt.imshow(
    ct_image[:,:,slice_id],
    cmap="gray"
)

plt.title("CT Image")

plt.axis("off")


plt.subplot(1,2,2)

plt.imshow(
    mask[:,:,slice_id]
)

plt.title("Ground Truth Mask")

plt.axis("off")

plt.show()
